## Task 6: Feature Selection and Recommendation Quality Evaluation

### Objective
To experiment with different combinations of textual features and identify which combination produces the most relevant TED Talk recommendations.

In this task, four experiments will be performed using TF-IDF vectorization and cosine similarity.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the cleaned TED Talks dataset
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

# Display dataset information
print("Dataset Shape:", df.shape)

print("\nAvailable Columns:")
print(df.columns.tolist())

# Display first five records
df.head()

Dataset Shape: (4005, 26)

Available Columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'description_clean', 'transcript_clean', 'topics_clean', 'combined_text', 'combined_text_length', 'combined_text_word_count']


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,url,description,transcript,title_clean,description_clean,transcript_clean,topics_clean,combined_text,combined_text_length,combined_text_word_count
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,...,https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",averting the climate crisis,with the same humor and humanity he exuded in ...,thank you so much chris and it s truly a great...,alternative energy cars climate change culture...,averting the climate crisis alternative energy...,11819,2269
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,...,https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac...",the best stats you ve ever seen,you ve never seen data presented like this wit...,about 10 years ago i took on the task to teach...,africa asia google demo economics global issue...,the best stats you ve ever seen africa asia go...,17185,3278
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,...,https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",simplicity sells,new york times columnist david pogue takes aim...,music the sound of silence simon garfunkel hel...,computers entertainment interface design media...,simplicity sells computers entertainment inter...,17892,3548
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...,greening the ghetto,in an emotionally charged talk macarthur winni...,if you re here today and i m very happy that y...,macarthur grant activism business cities envir...,greening the ghetto macarthur grant activism b...,18150,3226
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,...,https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,do schools kill creativity,sir ken robinson makes an entertaining and pro...,good morning how are you audience good it s be...,children creativity culture dance education pa...,do schools kill creativity children creativity...,17061,3326


In [2]:
df.columns.tolist()

['talk_id',
 'title',
 'speaker_1',
 'all_speakers',
 'occupations',
 'about_speakers',
 'views',
 'recorded_date',
 'published_date',
 'event',
 'native_lang',
 'available_lang',
 'comments',
 'duration',
 'topics',
 'related_talks',
 'url',
 'description',
 'transcript',
 'title_clean',
 'description_clean',
 'transcript_clean',
 'topics_clean',
 'combined_text',
 'combined_text_length',
 'combined_text_word_count']

In [3]:
# Select five TED Talks for recommendation evaluation
selected_talks = df["title"].dropna().drop_duplicates().head(5).tolist()

print("Selected TED Talks:")
for i, talk in enumerate(selected_talks, start=1):
    print(f"{i}. {talk}")

Selected TED Talks:
1. Averting the climate crisis
2. The best stats you've ever seen
3. Simplicity sells
4. Greening the ghetto
5. Do schools kill creativity?


In [4]:
# Check the feature columns required for Task 6
feature_columns = [
    "cleaned_title",
    "cleaned_description",
    "cleaned_transcript",
    "cleaned_topics"
]

for column in feature_columns:
    if column in df.columns:
        print(f"{column}: Available")
    else:
        print(f"{column}: Not found")

cleaned_title: Not found
cleaned_description: Not found
cleaned_transcript: Not found
cleaned_topics: Not found


In [5]:
# Display all available dataset columns
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

1. talk_id
2. title
3. speaker_1
4. all_speakers
5. occupations
6. about_speakers
7. views
8. recorded_date
9. published_date
10. event
11. native_lang
12. available_lang
13. comments
14. duration
15. topics
16. related_talks
17. url
18. description
19. transcript
20. title_clean
21. description_clean
22. transcript_clean
23. topics_clean
24. combined_text
25. combined_text_length
26. combined_text_word_count


In [6]:
# Create text combinations for all four experiments

# Experiment 1: Title only
df["exp_1_text"] = df["title_clean"].fillna("")

# Experiment 2: Title + Tags
df["exp_2_text"] = (
    df["title_clean"].fillna("") + " " +
    df["topics_clean"].fillna("")
)

# Experiment 3: Title + Description + Tags
df["exp_3_text"] = (
    df["title_clean"].fillna("") + " " +
    df["description_clean"].fillna("") + " " +
    df["topics_clean"].fillna("")
)

# Experiment 4: Title + Description + Transcript + Tags
df["exp_4_text"] = (
    df["title_clean"].fillna("") + " " +
    df["description_clean"].fillna("") + " " +
    df["transcript_clean"].fillna("") + " " +
    df["topics_clean"].fillna("")
)

print("All four feature combinations created successfully!")

print("\nFeature Combination Details:")
print("Experiment 1: Title only")
print("Experiment 2: Title + Tags")
print("Experiment 3: Title + Description + Tags")
print("Experiment 4: Title + Description + Transcript + Tags")

All four feature combinations created successfully!

Feature Combination Details:
Experiment 1: Title only
Experiment 2: Title + Tags
Experiment 3: Title + Description + Tags
Experiment 4: Title + Description + Transcript + Tags


In [7]:
def generate_recommendations(text_column, talk_title, top_n=5):

    # Create TF-IDF vectors
    vectorizer = TfidfVectorizer(
        max_features=5000,
        stop_words="english"
    )

    tfidf_matrix = vectorizer.fit_transform(df[text_column])

    # Calculate cosine similarity
    similarity_matrix = cosine_similarity(tfidf_matrix)

    # Find the selected TED Talk
    talk_index = df.index[df["title"] == talk_title][0]

    # Get similarity scores
    similarity_scores = list(enumerate(similarity_matrix[talk_index]))

    # Sort by similarity score
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Exclude the selected talk itself
    recommendations = [
        item for item in similarity_scores
        if item[0] != talk_index
    ][:top_n]

    # Create recommendation DataFrame
    results = pd.DataFrame([
        {
            "Recommended TED Talk": df.iloc[index]["title"],
            "Similarity Score": round(score, 4)
        }
        for index, score in recommendations
    ])

    return results

In [8]:
# Define the four experiments
experiments = {
    "Experiment 1: Title Only": "exp_1_text",
    "Experiment 2: Title + Tags": "exp_2_text",
    "Experiment 3: Title + Description + Tags": "exp_3_text",
    "Experiment 4: Title + Description + Transcript + Tags": "exp_4_text"
}

# Store all recommendation results
all_results = []

for experiment_name, text_column in experiments.items():

    print(f"\nRunning {experiment_name}...")

    # Generate TF-IDF vectors
    vectorizer = TfidfVectorizer(
        max_features=5000,
        stop_words="english"
    )

    tfidf_matrix = vectorizer.fit_transform(df[text_column])

    # Generate recommendations for each selected talk
    for talk_title in selected_talks:

        talk_index = df.index[df["title"] == talk_title][0]

        # Calculate similarity with the selected talk
        similarity_scores = cosine_similarity(
            tfidf_matrix,
            tfidf_matrix[talk_index]
        ).flatten()

        # Sort by similarity score
        similar_indices = np.argsort(similarity_scores)[::-1]

        # Exclude the selected TED Talk
        similar_indices = [
            i for i in similar_indices if i != talk_index
        ][:5]

        # Store the top five recommendations
        for rank, index in enumerate(similar_indices, start=1):
            all_results.append({
                "Experiment": experiment_name,
                "Selected TED Talk": talk_title,
                "Rank": rank,
                "Recommended TED Talk": df.iloc[index]["title"],
                "Similarity Score": round(
                    similarity_scores[index], 4
                )
            })

    print("Completed!")

# Convert results into a DataFrame
results_df = pd.DataFrame(all_results)

print("\nAll experiments completed successfully!")
print("Total recommendations generated:", len(results_df))

results_df.head(10)


Running Experiment 1: Title Only...
Completed!

Running Experiment 2: Title + Tags...
Completed!

Running Experiment 3: Title + Description + Tags...
Completed!

Running Experiment 4: Title + Description + Transcript + Tags...
Completed!

All experiments completed successfully!
Total recommendations generated: 100


,Experiment,Selected TED Talk,Rank,Recommended TED Talk,Similarity Score
0,Experiment 1: Title Only,Averting the climate crisis,1,New thinking on the climate crisis,0.4880
1,Experiment 1: Title Only,Averting the climate crisis,2,The global learning crisis -- and what to do a...,0.2934
2,Experiment 1: Title Only,Averting the climate crisis,3,Why you should be a climate activist,0.2854
3,Experiment 1: Title Only,Averting the climate crisis,4,The state of the climate — and what we might d...,0.2854
4,Experiment 1: Title Only,Averting the climate crisis,5,Why I must speak out about climate change,0.2617
5,Experiment 1: Title Only,The best stats you've ever seen,1,The best kindergarten you’ve ever seen,0.6920
6,Experiment 1: Title Only,The best stats you've ever seen,2,The sea we've hardly seen,0.4623
7,Experiment 1: Title Only,The best stats you've ever seen,3,Cooking as never seen before,0.3712
8,Experiment 1: Title Only,The best stats you've ever seen,4,Turning powerful stats into art,0.3097
9,Experiment 1: Title Only,The best stats you've ever seen,5,Surprising stats about child carseats,0.2954


In [9]:
# Display recommendations grouped by selected TED Talk
for talk_title in selected_talks:

    print("\n" + "=" * 100)
    print("SELECTED TED TALK:", talk_title)
    print("=" * 100)

    talk_results = results_df[
        results_df["Selected TED Talk"] == talk_title
    ]

    for experiment_name in experiments.keys():

        print(f"\n{experiment_name}")

        experiment_results = talk_results[
            talk_results["Experiment"] == experiment_name
        ]

        display(
            experiment_results[
                ["Rank", "Recommended TED Talk", "Similarity Score"]
            ].reset_index(drop=True)
        )


SELECTED TED TALK: Averting the climate crisis

Experiment 1: Title Only


,Rank,Recommended TED Talk,Similarity Score
0,1,New thinking on the climate crisis,0.4880
1,2,The global learning crisis -- and what to do a...,0.2934
2,3,Why you should be a climate activist,0.2854
3,4,The state of the climate — and what we might d...,0.2854
4,5,Why I must speak out about climate change,0.2617



Experiment 2: Title + Tags


,Rank,Recommended TED Talk,Similarity Score
0,1,New thinking on the climate crisis,0.4945
1,2,The case for optimism on climate change,0.4458
2,3,The future of cars,0.4378
3,4,Fusion is energy's future,0.4353
4,5,Why I must speak out about climate change,0.4335



Experiment 3: Title + Description + Tags


,Rank,Recommended TED Talk,Similarity Score
0,1,New thinking on the climate crisis,0.3914
1,2,What comes after An Inconvenient Truth?,0.3739
2,3,The case for optimism on climate change,0.3424
3,4,Why I must speak out about climate change,0.3144
4,5,The state of the climate — and what we might d...,0.3090


... (30 outputs left out)


Experiment 3: Title + Description + Tags


,Rank,Recommended TED Talk,Similarity Score
0,1,Bring on the learning revolution!,0.4362
1,2,How to escape education's death valley,0.3058
2,3,The child-driven education,0.1934
3,4,The orchestra in my mouth,0.1909
4,5,Use data to build better schools,0.1823



Experiment 4: Title + Description + Transcript + Tags


,Rank,Recommended TED Talk,Similarity Score
0,1,Bring on the learning revolution!,0.4921
1,2,How to escape education's death valley,0.4670
2,3,Nerdcore comedy,0.4101
3,4,A one-man world summit,0.4070
4,5,How to run a company with (almost) no rules,0.3758


In [10]:
# Compare recommendations across feature combinations

comparison_results = []

baseline_experiment = "Experiment 1: Title Only"

for talk_title in selected_talks:

    baseline = set(
        results_df[
            (results_df["Selected TED Talk"] == talk_title) &
            (results_df["Experiment"] == baseline_experiment)
        ]["Recommended TED Talk"]
    )

    for experiment_name in list(experiments.keys())[1:]:

        current = set(
            results_df[
                (results_df["Selected TED Talk"] == talk_title) &
                (results_df["Experiment"] == experiment_name)
            ]["Recommended TED Talk"]
        )

        comparison_results.append({
            "Selected TED Talk": talk_title,
            "Compared Experiment": experiment_name,
            "Common Recommendations": len(baseline & current),
            "New Recommendations": len(current - baseline),
            "Changed Recommendations": len(baseline - current)
        })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

,Selected TED Talk,Compared Experiment,Common Recommendations,New Recommendations,Changed Recommendations
0,Averting the climate crisis,Experiment 2: Title + Tags,2,3,3
1,Averting the climate crisis,Experiment 3: Title + Description + Tags,3,2,2
2,Averting the climate crisis,Experiment 4: Title + Description + Transcript...,1,4,4
3,The best stats you've ever seen,Experiment 2: Title + Tags,1,4,4
4,The best stats you've ever seen,Experiment 3: Title + Description + Tags,0,5,5
5,The best stats you've ever seen,Experiment 4: Title + Description + Transcript...,0,5,5
6,Simplicity sells,Experiment 2: Title + Tags,2,3,3
7,Simplicity sells,Experiment 3: Title + Description + Tags,2,3,3
8,Simplicity sells,Experiment 4: Title + Description + Transcript...,1,4,4
9,Greening the ghetto,Experiment 2: Title + Tags,0,5,5


In [11]:
# Compare Experiment 3 and Experiment 4

transcript_comparison = []

for talk_title in selected_talks:

    exp3 = set(
        results_df[
            (results_df["Selected TED Talk"] == talk_title) &
            (results_df["Experiment"] ==
             "Experiment 3: Title + Description + Tags")
        ]["Recommended TED Talk"]
    )

    exp4 = set(
        results_df[
            (results_df["Selected TED Talk"] == talk_title) &
            (results_df["Experiment"] ==
             "Experiment 4: Title + Description + Transcript + Tags")
        ]["Recommended TED Talk"]
    )

    transcript_comparison.append({
        "Selected TED Talk": talk_title,
        "Common Recommendations": len(exp3 & exp4),
        "New Recommendations After Adding Transcript": len(exp4 - exp3),
        "Recommendations Removed": len(exp3 - exp4)
    })

transcript_comparison_df = pd.DataFrame(transcript_comparison)

display(transcript_comparison_df)

,Selected TED Talk,Common Recommendations,New Recommendations After Adding Transcript,Recommendations Removed
0,Averting the climate crisis,1,4,4
1,The best stats you've ever seen,3,2,2
2,Simplicity sells,1,4,4
3,Greening the ghetto,2,3,3
4,Do schools kill creativity?,2,3,3


In [12]:
# Display recommendations in a compact comparison table

for talk_title in selected_talks:

    print("\nSelected TED Talk:", talk_title)

    comparison_table = results_df[
        results_df["Selected TED Talk"] == talk_title
    ].pivot(
        index="Rank",
        columns="Experiment",
        values="Recommended TED Talk"
    )

    display(comparison_table)


Selected TED Talk: Averting the climate crisis


Experiment,Experiment 1: Title Only,Experiment 2: Title + Tags,Experiment 3: Title + Description + Tags,Experiment 4: Title + Description + Transcript + Tags
Rank,,,,
1,New thinking on the climate crisis,New thinking on the climate crisis,New thinking on the climate crisis,New thinking on the climate crisis
2,The global learning crisis -- and what to do a...,The case for optimism on climate change,What comes after An Inconvenient Truth?,How we can turn the tide on climate
3,Why you should be a climate activist,The future of cars,The case for optimism on climate change,A climate solution where all sides can win
4,The state of the climate — and what we might d...,Fusion is energy's future,Why I must speak out about climate change,A one-man world summit
5,Why I must speak out about climate change,Why I must speak out about climate change,The state of the climate — and what we might d...,How we could change the planet's climate future



Selected TED Talk: The best stats you've ever seen


Experiment,Experiment 1: Title Only,Experiment 2: Title + Tags,Experiment 3: Title + Description + Tags,Experiment 4: Title + Description + Transcript + Tags
Rank,,,,
1,The best kindergarten you’ve ever seen,New insights on poverty,Let my dataset change your mindset,Let my dataset change your mindset
2,The sea we've hardly seen,Let my dataset change your mindset,New insights on poverty,New insights on poverty
3,Cooking as never seen before,Asia's rise -- how and when,The good news of the decade? We're winning the...,The good news of the decade? We're winning the...
4,Turning powerful stats into art,The global goals we've made progress on -- and...,Asia's rise -- how and when,"Insights on HIV, in stunning data visuals"
5,Surprising stats about child carseats,The best kindergarten you’ve ever seen,How not to be ignorant about the world,"Global population growth, box by box"



Selected TED Talk: Simplicity sells


Experiment,Experiment 1: Title Only,Experiment 2: Title + Tags,Experiment 3: Title + Description + Tags,Experiment 4: Title + Description + Transcript + Tags
Rank,,,,
1,Toward a science of simplicity,Designing for simplicity,Toward a science of simplicity,Nerdcore comedy
2,Designing for simplicity,Toward a science of simplicity,Designing for simplicity,Design and discovery
3,The dark history of IQ tests,Tidying up art,Rethink the desktop with BumpTop,A comic sendup of TED2006
4,"Crisis support for the world, one text away",Software (as) art,"5 predictions, from 1984",Cool tricks your phone can do
5,A global pandemic calls for global solutions,"Less stuff, more happiness",The music wars,Designing for simplicity



Selected TED Talk: Greening the ghetto


Experiment,Experiment 1: Title Only,Experiment 2: Title + Tags,Experiment 3: Title + Description + Tags,Experiment 4: Title + Description + Transcript + Tags
Rank,,,,
1,The electrifying speeches of Sojourner Truth,Winning the oil endgame,Winning the oil endgame,"The next big thing is coming from the Bronx, a..."
2,"How ""policing for profit"" undermines your rights","Rebuilding a neighborhood with beauty, dignity...",Everyday inventions,A teacher growing green in the South Bronx
3,The dark history of IQ tests,An illustrated journey through Rome,A teacher growing green in the South Bronx,How students of color confront impostor syndrome
4,"Crisis support for the world, one text away",Juggling as art ... and science,"Rebuilding a neighborhood with beauty, dignity...",3 stories of local eco-entrepreneurship
5,A global pandemic calls for global solutions,Buildings that blend nature and city,3 stories of local eco-entrepreneurship,How public spaces make cities work



Selected TED Talk: Do schools kill creativity?


Experiment,Experiment 1: Title Only,Experiment 2: Title + Tags,Experiment 3: Title + Description + Tags,Experiment 4: Title + Description + Transcript + Tags
Rank,,,,
1,4 lessons in creativity,Our failing schools. Enough is enough!,Bring on the learning revolution!,Bring on the learning revolution!
2,Our failing schools. Enough is enough!,Creative ways to get kids to thrive in school,How to escape education's death valley,How to escape education's death valley
3,Fashion and creativity,A father-daughter dance ... in prison,The child-driven education,Nerdcore comedy
4,Don't kill your language,Kids need structure,The orchestra in my mouth,A one-man world summit
5,Is space trying to kill us?,Use data to build better schools,Use data to build better schools,How to run a company with (almost) no rules


In [13]:
# Qualitative evaluation of the four experiments

evaluation_data = [
    {
        "Experiment": "Title Only",
        "Observation": "Recommendations are often influenced by similar title words.",
        "Relevance": "Low to Moderate"
    },
    {
        "Experiment": "Title + Tags",
        "Observation": "Tags improve topic matching and introduce more subject-related recommendations.",
        "Relevance": "Moderate to Good"
    },
    {
        "Experiment": "Title + Description + Tags",
        "Observation": "Descriptions provide additional context and generally produce balanced recommendations.",
        "Relevance": "Good"
    },
    {
        "Experiment": "Title + Description + Transcript + Tags",
        "Observation": "Transcripts add detailed information but may introduce unrelated words and recommendations.",
        "Relevance": "Mixed"
    }
]

evaluation_df = pd.DataFrame(evaluation_data)

display(evaluation_df)

,Experiment,Observation,Relevance
0,Title Only,Recommendations are often influenced by simila...,Low to Moderate
1,Title + Tags,Tags improve topic matching and introduce more...,Moderate to Good
2,Title + Description + Tags,Descriptions provide additional context and ge...,Good
3,Title + Description + Transcript + Tags,Transcripts add detailed information but may i...,Mixed


## Final Feature Selection and Justification

After evaluating four different feature combinations using five selected TED Talks, Experiment 3 (Title + Description + Tags) was selected as the preferred feature combination.

### Reasons for Selection

1. **Title:** Provides the main subject and central idea of the TED Talk.
2. **Description:** Adds contextual information and helps identify the actual topic.
3. **Tags:** Provide important keywords that improve topic matching.

### Why Not the Other Combinations?

- **Title Only:** Relies heavily on matching words in titles and may miss the actual subject.
- **Title + Tags:** Improves recommendations but provides less contextual information than descriptions.
- **Title + Description + Transcript + Tags:** Transcripts provide additional details but may introduce unrelated words and reduce recommendation relevance in some cases.

### Conclusion

Experiment 3 provides a good balance between topic relevance and contextual information. Therefore, Title + Description + Tags is selected as the preferred feature combination for the TED Talks recommendation system.

This evaluation was based on qualitative inspection of recommendations for five selected TED Talks. Further testing with more talks and user feedback would help validate the selection.

## Overall Observations

1. Different feature combinations produce different TED Talk recommendations.
2. Title-only recommendations are mainly influenced by similarities between title words.
3. Adding tags improves recommendations by including topic-related keywords.
4. Adding descriptions provides more contextual information and improves topic matching.
5. Transcripts significantly change recommendation results because they contain detailed language and additional topics.
6. More features do not always guarantee better recommendations.

## Limitations

1. The evaluation was performed using only five selected TED Talks.
2. Recommendation relevance was evaluated manually rather than using a formal relevance metric.
3. TF-IDF focuses on word frequency and does not fully understand the meaning or context of sentences.
4. Long transcripts may introduce noise and unrelated terms.
5. Different users may have different preferences, which were not considered in this content-based evaluation.

## Final Conclusion

The experiments demonstrate that feature selection plays an important role in content-based recommendation systems.

Among the four combinations, Title + Description + Tags was selected as the preferred combination because it provided a good balance of topic relevance and contextual information in the evaluated examples.